# z303 — Etapa 3: Optuna — Búsqueda Bayesiana

Input  : `z302_features.parquet`
Output : `z303_hiperparametros.json`

Responsabilidades:
- Espacio de búsqueda de LightGBM con Optuna (TPE = Bayesiana)
- Validación temporal sin leakage
- Features categóricas declaradas a LGBM
- **WAPE siempre medido sobre el NIVEL en toneladas** (reconstruye si target=delta)
- `tipo_target` se lee del dataset y se propaga al JSON
- Storage SQLite (trials acumulables)

## 0. Ambiente

In [ ]:
import os, shutil, subprocess

BASE       = '/home/ds/buckets/b1'
LOCAL_HOME = '/home/ds'
os.makedirs(f'{BASE}/exp', exist_ok=True)

# Kaggle auth: usar el de ~/.kaggle si ya existe; si no, buscarlo en el bucket
kaggle_dst = os.path.expanduser('~/.kaggle/kaggle.json')
os.makedirs(os.path.dirname(kaggle_dst), exist_ok=True)
if os.path.exists(kaggle_dst):
    os.chmod(kaggle_dst, 0o600)
    print('Kaggle auth OK (ya estaba en ~/.kaggle)')
else:
    _encontrado = False
    for cand in [f'{BASE}/kaggle.json', f'{BASE}/kaggle/kaggle.json']:
        if os.path.exists(cand):
            shutil.copy(cand, kaggle_dst)
            os.chmod(kaggle_dst, 0o600)
            print(f'Kaggle auth OK (copiado de {cand})')
            _encontrado = True
            break
    if not _encontrado:
        print('⚠️  kaggle.json no encontrado.')

In [ ]:
!pip install -q uv
!uv pip install -q pyarrow lightgbm optuna sqlalchemy

## 1. Parámetros — palancas

In [ ]:
import os

PARAM = {
    'experimento': 'z303',

    # ── PALANCA 1 (debe coincidir con z301/z302) ────────────────────
    'modo_agrupacion': 'producto',

    # ── PALANCA 8: trials nuevos por corrida ─────────────────────────
    'n_trials': 50,

    # ── PALANCA 9: esquema de validación ─────────────────────────
    'esquema_val': 'ultimo_periodo',
    'walk_forward_k': 3,

    # ── PALANCA 10: métrica ─────────────────────────────────
    'metrica': 'wape',

    # ── PALANCA 11: sampling de filas ───────────────────────────
    'sampling_frac': None,

    # ── PALANCA 12: objetivo de LGBM ───────────────────────────
    # 'regression' | 'tweedie' | 'poisson' | 'regression_l1'
    'objective_lgbm': 'regression',
    'tweedie_optimizar': True,

    # ── PALANCA 15: regularización ─────────────────────────────
    # 'normal' → rangos amplios.  'fuerte' → árboles chicos, más regularización
    'regularizacion': 'normal',

    # ── PALANCA 16: peso por recencia ──────────────────────────
    # None → todos los períodos pesan igual
    # float → decay (0.9 = cada mes hacia atrás pesa 0.9x el siguiente)
    'decay_recencia': None,

    # ── PALANCA 5: tipo de target (elige la columna) ─────────────────
    # 'nivel' → usa target_nivel.   'delta' → usa target_delta
    'tipo_target': 'nivel',

    # ── PALANCA 19: features a EXCLUIR ──────────────────────────
    # Por defecto se usan TODAS las features del dataset.
    # Poné acá los nombres de columnas que querés sacar en este experimento.
    # Ej: ['anio', 'cat1', 'ms_delta_3']
    'features_excluir': [],

    'semilla': 102191,

    'cols_categoricas': ['cat1', 'cat2', 'cat3', 'brand'],
}

# Paths e identificadores derivados del modo → cada modo tiene su study y su JSON
MODO = PARAM['modo_agrupacion']
PARAM['path_input']   = f'/home/ds/buckets/b1/exp/z302_features_{MODO}.parquet'
PARAM['path_output']  = f"/home/ds/buckets/b1/exp/z303_hiper_{MODO}_{PARAM['experimento']}.json"
PARAM['path_storage'] = f'sqlite:////home/ds/z303_optuna_{MODO}.db'
PARAM['study_name']   = f"{PARAM['experimento']}_{MODO}"

ruta_exp = '/home/ds/buckets/b1/exp/' + PARAM['experimento']
os.makedirs(ruta_exp, exist_ok=True)
os.chdir(ruta_exp)
print('Parámetros:', PARAM)
print('Input:',  PARAM['path_input'])
print('Study:',  PARAM['study_name'])

## 2. Carga y features

Lee `tipo_target` del dataset (lo propaga z302). `tn` y `tipo_target` no son features pero se conservan en `df_pd` para reconstruir el nivel.

In [ ]:
import polars as pl
import numpy as np
import lightgbm as lgb
import optuna
import json
optuna.logging.set_verbosity(optuna.logging.WARNING)

df = pl.read_parquet(PARAM['path_input'])
print(f'Dataset: {df.shape}')

# Tipo de target (palanca) → elige la columna
TIPO_TARGET = PARAM['tipo_target']
TARGET_COL  = 'target_delta' if TIPO_TARGET == 'delta' else 'target_nivel'
print(f'Tipo de target: {TIPO_TARGET}  (columna: {TARGET_COL})')

# Columnas que NUNCA son features (ids, target, metadata, tn actual = lag_0)
COLS_EXCLUIR_BASE = [
    'agrupa_id', 'product_id', 'customer_id', 'periodo',
    'tn', 'tn_t2', 'target_nivel', 'target_delta',
    'modo_agrupacion', 'solo_predecir', 'tipo_target',
]

# Features = todas menos las base menos las que el experimento excluye (palanca 19)
FEATURES = [c for c in df.columns
            if c not in COLS_EXCLUIR_BASE and c not in PARAM['features_excluir']]
CAT_FEATURES = [c for c in PARAM['cols_categoricas'] if c in FEATURES]

print(f'Features usadas ({len(FEATURES)}): {FEATURES}')
if PARAM['features_excluir']:
    print(f'Features excluidas: {PARAM["features_excluir"]}')
print(f'Categóricas: {CAT_FEATURES}')

## 3. Métrica

WAPE sobre niveles en toneladas. Recibe `y_real_nivel` y `y_pred_nivel` ya reconstruidos.

In [ ]:
def calcular_metrica(y_real, y_pred, metrica='wape'):
    y_real = np.array(y_real, dtype=np.float64)
    y_pred = np.maximum(np.array(y_pred, dtype=np.float64), 0.0)
    if metrica == 'wape':
        den = y_real.sum()
        return np.nan if den == 0 else np.abs(y_real - y_pred).sum() / den
    elif metrica == 'mae':
        return np.abs(y_real - y_pred).mean()
    raise ValueError(f'Métrica desconocida: {metrica}')

## 4. Validación temporal

In [ ]:
periodos_ordenados = sorted(df['periodo'].unique().to_list())
print(f'Períodos: {periodos_ordenados[0]} → {periodos_ordenados[-1]}')

def get_splits(periodos, esquema, k=3):
    if esquema == 'ultimo_periodo':
        return [(periodos[-2], periodos[-1])]
    elif esquema == 'walk_forward_k':
        splits = []
        for i in range(k, 0, -1):
            splits.append((periodos[-(i + 1)], periodos[-i]))
        return splits
    raise ValueError(f'Esquema desconocido: {esquema}')

splits = get_splits(periodos_ordenados, PARAM['esquema_val'], PARAM['walk_forward_k'])
print('Splits:')
for corte, val_p in splits:
    print(f'  Train ≤ {corte}  |  Val = {val_p}')

## 5. Función objetivo

Reconstrucción del nivel:
- target='nivel' → pred_nivel = pred,  real_nivel = target
- target='delta' → pred_nivel = tn + pred,  real_nivel = tn + target  (= tn_t2)

El WAPE se calcula siempre sobre el nivel, igual que Kaggle.

In [ ]:
df_pd = df.to_pandas()
for c in CAT_FEATURES:
    df_pd[c] = df_pd[c].astype('category')

def calcular_pesos(periodos_serie, decay):
    """Peso por recencia: el período más reciente pesa 1, cada mes hacia atrás decae."""
    if decay is None:
        return None
    periodos = sorted(periodos_serie.unique())
    idx = {p: i for i, p in enumerate(periodos)}
    n = len(periodos)
    return periodos_serie.map(lambda p: decay ** (n - 1 - idx[p])).values

def espacio_hiper(trial):
    """Rangos de hiperparámetros según la palanca de regularización."""
    base = {
        'objective':     PARAM['objective_lgbm'],
        'metric':        'mae',
        'verbosity':     -1,
        'boosting_type': 'gbdt',
        'seed':          PARAM['semilla'],
        'subsample_freq':1,
    }
    if PARAM['regularizacion'] == 'fuerte':
        base.update({
            'num_leaves':       trial.suggest_int('num_leaves', 8, 64),
            'max_depth':        trial.suggest_int('max_depth', 3, 7),
            'learning_rate':    trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
            'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
            'min_child_samples':trial.suggest_int('min_child_samples', 30, 200),
            'subsample':        trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
            'reg_alpha':        trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        })
    else:  # 'normal'
        base.update({
            'num_leaves':       trial.suggest_int('num_leaves', 20, 300),
            'max_depth':        trial.suggest_int('max_depth', 3, 12),
            'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'n_estimators':     trial.suggest_int('n_estimators', 100, 2000),
            'min_child_samples':trial.suggest_int('min_child_samples', 5, 100),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        })
    if PARAM['objective_lgbm'] == 'tweedie' and PARAM.get('tweedie_optimizar', False):
        base['tweedie_variance_power'] = trial.suggest_float('tweedie_variance_power', 1.1, 1.9)
    return base

def objective(trial):
    params = espacio_hiper(trial)

    errores = []
    for corte, val_p in splits:
        df_tr = df_pd[df_pd['periodo'] <= corte].copy()
        df_vl = df_pd[df_pd['periodo'] == val_p].copy()
        if len(df_vl) == 0:
            continue
        if PARAM['sampling_frac'] is not None:
            df_tr = df_tr.sample(frac=PARAM['sampling_frac'], random_state=PARAM['semilla'])

        X_tr, y_tr = df_tr[FEATURES], df_tr[TARGET_COL].values
        X_vl, y_vl = df_vl[FEATURES], df_vl[TARGET_COL].values
        w_tr = calcular_pesos(df_tr['periodo'], PARAM['decay_recencia'])

        modelo = lgb.LGBMRegressor(**params)
        modelo.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            eval_set=[(X_vl, y_vl)],
            categorical_feature=CAT_FEATURES,
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        )
        pred = modelo.predict(X_vl)

        # Reconstruir nivel para medir WAPE como Kaggle
        if TIPO_TARGET == 'delta':
            tn_actual  = df_vl['tn'].values
            pred_nivel = tn_actual + pred
            real_nivel = tn_actual + y_vl
        else:
            pred_nivel = pred
            real_nivel = y_vl

        errores.append(calcular_metrica(real_nivel, pred_nivel, PARAM['metrica']))

    return float(np.mean(errores))

## 6. Correr Optuna (storage persistente)

Los trials se acumulan en SQLite. Para empezar limpio, cambiá `experimento` en PARAM.

In [ ]:
from tqdm.auto import tqdm

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
    study_name=PARAM['study_name'],
    storage=PARAM['path_storage'],
    load_if_exists=True
)

print(f'Trials previos: {len(study.trials)}')
print(f'Corriendo {PARAM["n_trials"]} trials nuevos...')

with tqdm(total=PARAM['n_trials'], desc='Optuna') as pbar:
    def callback(study, trial):
        pbar.update(1)
        pbar.set_postfix({'mejor': f'{study.best_value:.4f}'})
    study.optimize(objective, n_trials=PARAM['n_trials'], callbacks=[callback])

print(f'\n✅ Optuna finalizado.')
print(f'   Trials totales: {len(study.trials)}')
print(f'   Mejor {PARAM["metrica"]} (nivel): {study.best_value:.4f}')
print(f'   Mejores hiperparámetros: {study.best_params}')

## 7. Guardar resultados

Guarda `tipo_target` para que z304 reconstruya solo.

In [ ]:
resultado = {
    'experimento':      PARAM['experimento'],
    'modo_agrupacion':  PARAM['modo_agrupacion'],
    'metrica':          PARAM['metrica'],
    'mejor_valor':      study.best_value,
    'n_trials_total':   len(study.trials),
    'esquema_val':      PARAM['esquema_val'],
    'tipo_target':      TIPO_TARGET,
    'objective_lgbm':   PARAM['objective_lgbm'],
    'regularizacion':   PARAM['regularizacion'],
    'decay_recencia':   PARAM['decay_recencia'],
    'features':         FEATURES,
    'cat_features':     CAT_FEATURES,
    'hiperparametros':  study.best_params,
}

with open(PARAM['path_output'], 'w') as f:
    json.dump(resultado, f, indent=2)

print(f'✅ Guardado: {PARAM["path_output"]}')
print(json.dumps(resultado, indent=2))

# Backup del study al bucket (el .db local se pierde si se destruye la VM)
import shutil
db_local  = PARAM['path_storage'].replace('sqlite:///', '')
db_bucket = f"/home/ds/buckets/b1/exp/z303_optuna_{PARAM['modo_agrupacion']}.db"
shutil.copy(db_local, db_bucket)
print(f'✅ Backup del study: {db_bucket}')

## 8. Visualización (opcional)

In [ ]:
try:
    import optuna.visualization as vis
    vis.plot_param_importances(study).show()
except Exception as e:
    print(f'Visualización no disponible: {e}')

df_trials = study.trials_dataframe()
print(df_trials[['number', 'value', 'state']].sort_values('value').head(10))